# Rekenkamer in Parliamentary Discourse — Analysis

This notebook traces how the Dutch Court of Audit (*Rekenkamer*) appears in parliamentary debates
from the 19th century to the present, and situates those references within the thematic structure
of Dutch parliamentary debate as captured by LDA topic models.

**Structure**
1. Setup & data loading
2. Long-run frequency of references (since 1849)
3. Seasonal patterns by week of year
4. Who references the Rekenkamer — government or opposition?
5. Topic–reference partial correlations (300-topic model)
6. Evaluation / oversight topics over time (500-topic model)
7. Verantwoordingsdag — does accountability day spike?
8. Pairwise topic co-occurrence around the focal topic

**Data paths** are resolved via `paths.py`.  Two LDA models are used:
- **300-topic** (1972–2022, `REKENKAMER_LDA_DIR`): used for partial-correlation analysis  
- **500-topic** (1945–2022, `REKENKAMER_LDA_DIR_500`): used for control-topic and Verantwoordingsdag analysis

## 1  Setup

In [ ]:
from helper import *
from paths import REFERENCES_CSV, LDA_DIR, LDA_DIR_500, NOUN_TOTAL_CSV, CABINETS_CSV

In [ ]:
refs, dist, meta, ks, total_nouns, cab = load_all(
    references_path=REFERENCES_CSV,
    lda_dir=LDA_DIR,
    noun_total_path=NOUN_TOTAL_CSV,
    cabinet_path=CABINETS_CSV,
)

TOPIC_COLS = list(range(300))

refs_commons = refs[refs["house"] == "commons"].copy()
refs_commons, dist = add_periods(refs_commons, dist)

dist_averaged = dist.groupby(dist["date"].dt.to_period("Q"))[TOPIC_COLS].mean()

## 2  Long-run frequency of references

How many times did speakers mention the Court of Audit, and how did this change over time?
The **red area** shows summed relative frequency (normalised by total noun count per day);
the **blue line** shows entropy — how spread out the references are across days within each quarter.

High entropy + high frequency = the Rekenkamer is discussed routinely by many speakers on many days.  
Low entropy + high frequency = a concentrated spike — one big debate or one landmark report.  
The grey band marks the German occupation (1940–1945), when parliament was suspended.

In [ ]:
daily = relative_frequency(refs_commons, total_nouns)
quarterly = quarterly_stats(daily, lowess_frac=0.05)

fig = plot_freq_entropy(quarterly, war_shade=True)
fig.savefig("../results/figs/freq_entropy.png", dpi=150, bbox_inches="tight")

## 3  Seasonal patterns by week of year

Does the Rekenkamer follow the parliamentary calendar?  Each row is a 5-year bin;
each column is a week of the year.  Intensity = share of that bin's references in that week.

*Verantwoordingsdag* (third Wednesday of May, week ~20) and the autumn budget debates
(September–October, weeks 36–42) should show up as recurrent peaks.

In [ ]:
fig = plot_weekly_heatmap(refs_commons, year_start=1940, period_size=5)
fig.savefig("../results/figs/weekly_heatmap.png", dpi=150, bbox_inches="tight")

## 4  Who references the Rekenkamer — government or opposition?

Each speaker is classified as **government** (minister / state secretary), **coalition backbencher**,
or **opposition MP** based on which cabinet was in office on that date.

If the opposition drives references, the Rekenkamer functions as an accountability instrument.
If the government drives them, the Rekenkamer is more likely cited to *legitimise* existing policy.

In [ ]:
refs_classified = classify_coalition(refs_commons, cab)

fig = plot_coalition_area(refs_classified, year_start=1945)
fig.savefig("../results/figs/coalition_area.png", dpi=150, bbox_inches="tight")

## 5  Topic–reference partial correlations (300-topic model)

For each quarter we compute the partial correlation between each LDA topic's daily proportion
and the number of Rekenkamer references — *controlling for all other topics simultaneously*
via precision-matrix inversion.

A positive partial correlation means: on days when the corpus talks more about topic *T*, there are
also more Rekenkamer references, even after accounting for the general thematic ebb and flow.

In [ ]:
dam, refs_day = build_dam(dist, refs_commons)

In [ ]:
res = pcorr_over_time(dam, refs_day, min_obs=30)
print(f"{len(res):,} topic-quarter observations")

In [ ]:
res["label"] = res["topic"].astype(str) + " " + res["topic"].map(ks)
resp = res.groupby(["period", "label"])["partial_corr"].mean().unstack().dropna()

### 5.1  Topics with a rising association with the Rekenkamer

Mann-Kendall slope on the EWM-smoothed partial-correlation time series.
A positive slope means the topic has become *more* associated with Rekenkamer references over time.

In [ ]:
trending_topics(resp, n=15, ewm_span=12)

### 5.2  Topic landscape

**x** = variance over time (does the association fluctuate?),  
**y** = mean partial correlation (how persistently linked to the Rekenkamer?),  
**bubble size** = peak association,  
**colour** = overall corpus prominence (log₂-softmax z-score).

Top-right = strong *and* consistent association with the Rekenkamer.

In [ ]:
scatter_df = topic_scatter_data(resp, dist_averaged)
fig = plot_topic_bubble(scatter_df, ks, top_n_labels=12)
fig.savefig("../results/figs/topic_bubble.png", dpi=150, bbox_inches="tight")

### 5.3  Deep-dive: individual topics

**Blue** = quarterly partial correlation with Rekenkamer references.  
**Red dashed** = overall topic prevalence in the corpus (quarterly average).

When the lines diverge the link to the Rekenkamer reflects something specific about *oversight*
discourse — not just the general salience of the topic in parliament.

Edit `TOPICS_OF_INTEREST` to explore other topic IDs (`ks[t]` gives the keyword list).

In [ ]:
TOPICS_OF_INTEREST = [236, 195, 141, 77, 261]

for t in TOPICS_OF_INTEREST:
    fig = plot_topic_timeseries(resp, dist_averaged, t, ks, ewm_span=20)
    fig.savefig(f"../results/figs/topic_{t}_timeseries.png", dpi=150, bbox_inches="tight")

### 5.4  Selected topics — heatmap over time

Rows = topics from `TOPICS_OF_INTEREST`.  Columns = quarters.  
Red = strong positive association; blue = negative.

In [ ]:
fig = plot_topic_heatmap(resp, TOPICS_OF_INTEREST, ks, ewm_span=12)
fig.savefig("../results/figs/topic_heatmap.png", dpi=150, bbox_inches="tight")

## 6  Evaluation and oversight topics over time (500-topic model)

The 500-topic model trained on Handelingen 1945–2022 contains several topics that directly
capture evaluation, control, and financial oversight discourse.  Tracking their prevalence
over time shows whether *oversight* as a parliamentary activity has grown independently
of how often the Rekenkamer is explicitly named.

The control topics listed below were identified by keyword inspection; edit the list to explore others.

In [ ]:
dfd, ks500 = load_averaged_topics(LDA_DIR_500)

In [ ]:
# Topic IDs identified as evaluation / oversight topics in the 500-topic model
CONTROL_TOPICS = [78, 100, 253, 88, 317, 194]

fig = plot_control_topics(dfd, CONTROL_TOPICS, ks500, ewm_span=8)
fig.savefig("../results/figs/control_topics.png", dpi=150, bbox_inches="tight")

## 7  Verantwoordingsdag — does accountability day spike?

*Verantwoordingsdag* (the third Wednesday of May) is the annual accountability day in the Dutch
parliament: ministers account for the previous year's budget to the Tweede Kamer.  If the
Rekenkamer functions as an institution tied to the parliamentary budget cycle, we would expect
evaluation topics to be significantly more prominent on that day than on other days.

In [ ]:
vdag_results = verantwoordingsdag_test(dfd, CONTROL_TOPICS, year_range=range(2000, 2023))
vdag_results.sort_values("p_value")

## 8  Topic co-occurrence around a focal topic

Which topics consistently appear alongside the focal accountability/finance topic?
The heatmap shows how the topic's neighbourhood shifts across decades.

In [ ]:
FOCAL_TOPIC = 236        # financiën, begroting, verantwoording …
SCORE_THRESHOLD = 0.1

yearly_pairs = []
for y, ds in dist.groupby(dist["date"].dt.year):
    try:
        pc = topic_corr_matrix(ds[TOPIC_COLS])
        long = (
            pc.stack().rename("score").reset_index()
            .rename(columns={"level_0": "topic_i", "level_1": "topic_j"})
        )
        long = long[
            (long["topic_i"] != long["topic_j"]) &
            (long["score"] > SCORE_THRESHOLD)
        ]
        long["topic_i"] = long["topic_i"].map(ks)
        long["topic_j"] = long["topic_j"].map(ks)
        yearly_pairs.append(long.assign(year=y))
    except Exception:
        continue

pairs_df = pd.concat(yearly_pairs, ignore_index=True)

In [ ]:
focal_label = ks[FOCAL_TOPIC]
subset = (
    pairs_df[pairs_df["topic_i"] == focal_label]
    .groupby(["year", "topic_j"])["score"].mean()
    .unstack().fillna(0)
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(subset.T, ax=ax, cmap="Blues", cbar_kws={"shrink": 0.6})
ax.set_title(f"Topics co-occurring with: {focal_label[:60]}", loc="left")
fig.tight_layout()
fig.savefig(f"../results/figs/pairwise_{FOCAL_TOPIC}.png", dpi=150, bbox_inches="tight")